# HostWise — Track B: Preprocessing and Model Comparison

**Goal:** choose the shared preprocessing version, compare models fairly, and identify the strongest current candidate.

**Parts**
1. **Preprocessing comparison:** V1, V2, V2.1 with Linear Regression
2. **Model comparison:** Dummy, Linear Regression, Ridge, HistGradientBoosting
3. **Conclusions:** results, limitations, a new-listing prediction, instructor checklist

**Evaluation protocol (same everywhere)**
- Training set only (`train_ids_eur.csv`, 123,049 listings); test set not used
- Target: `price_eur`
- Preprocessing inside the pipeline, refitted in every fold
- Same 5 folds, stratified by city, `random_state=42`
- Metrics: MAE (primary), RMSE, R², MAE per city (EUR)
- Negative predictions clipped to 0

**Selection rules (set before seeing results)**
- **Preprocessing:** lowest mean CV MAE. If V2.1 is within 1 EUR of the best version, prefer V2.1 (recovers missing bathroom counts and adds bathroom privacy).
- **Model:** lowest mean CV MAE gives the strongest *candidate*, not the final model.

*These preprocessing versions were explored earlier in a separate notebook; all results here are recomputed.*

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline

from preprocessing import (
    preprocessor, FEATURES,
    preprocessor_v2, FEATURES_V2,
    preprocessor_v21, FEATURES_V21,
)

pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
DATA_PATH = Path("../data/processed/hostwise_5cities.csv")
SPLIT_DIR = Path("../data/splits")

CITIES = ["london", "paris", "rome", "madrid", "barcelona"]
TARGET = "price_eur"
GBP_TO_EUR = 1.15        # same fixed rate as the shared foundation
RANDOM_STATE = 42
MAE_TOLERANCE = 1.0      # EUR, preprocessing selection rule

VERSIONS = {
    "V1": (preprocessor, FEATURES),
    "V2": (preprocessor_v2, FEATURES_V2),
    "V2.1": (preprocessor_v21, FEATURES_V21),
}

## Shared training data
Rebuild `price_eur` as in the shared foundation and select the training rows by their saved IDs. The test IDs are not loaded.

In [ ]:
split_config = json.loads((SPLIT_DIR / "split_config_eur.json").read_text())
assert split_config["target"]["gbp_to_eur"] == GBP_TO_EUR, "Rate differs from the shared foundation"

df = pd.read_csv(DATA_PATH, low_memory=False, dtype={"id": "int64"}).set_index("id")
df["price_num"] = pd.to_numeric(df["price"].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce")
df[TARGET] = df["price_num"] * np.where(df["city"] == "london", GBP_TO_EUR, 1.0)

train_ids = pd.read_csv(SPLIT_DIR / "train_ids_eur.csv", dtype={"id": "int64"})["id"]
train_df = df.loc[train_ids]

assert len(train_df) == split_config["n_train"], "Training rows differ from the shared foundation"
assert train_df[TARGET].notna().all() and (train_df[TARGET] > 0).all(), "Invalid target values"
train_df["city"].value_counts().reindex(CITIES).to_frame("train listings")

## Cross-validation folds
The 5 folds are created once (stratified by city) and reused for every experiment, so all versions and models see exactly the same rows.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(train_df, train_df["city"]))

pd.DataFrame({
    f"fold {i + 1}": train_df["city"].iloc[val_idx].value_counts().reindex(CITIES)
    for i, (_, val_idx) in enumerate(folds)
})

## Pipeline and evaluation functions
- `make_full_pipeline`: preprocessing + model (fresh copies each time)
- `evaluate`: out-of-fold predictions on the shared folds, negative predictions clipped to 0, MAE/RMSE/R² per fold and MAE per city
- `summarize`: mean and standard deviation over the 5 folds

In [ ]:
def make_full_pipeline(preprocessing, model):
    return Pipeline([("preprocessor", clone(preprocessing)), ("model", clone(model))])


def evaluate(pipeline, features):
    X, y = train_df[features], train_df[TARGET]
    y_pred = np.clip(cross_val_predict(pipeline, X, y, cv=folds, n_jobs=-1), 0, None)
    fold_scores = pd.DataFrame([
        {
            "MAE": mean_absolute_error(y.iloc[idx], y_pred[idx]),
            "RMSE": mean_squared_error(y.iloc[idx], y_pred[idx]) ** 0.5,
            "R2": r2_score(y.iloc[idx], y_pred[idx]),
        }
        for _, idx in folds
    ])
    city_mae = (y - y_pred).abs().groupby(train_df["city"]).mean().reindex(CITIES)
    return fold_scores, city_mae


def summarize(results):
    return pd.DataFrame({
        name: {
            "MAE": s["MAE"].mean(), "MAE std": s["MAE"].std(),
            "RMSE": s["RMSE"].mean(), "RMSE std": s["RMSE"].std(),
            "R2": s["R2"].mean(),
        }
        for name, s in results.items()
    }).T

---
# Part 1 — Preprocessing Comparison
Same model (Linear Regression), folds and target for V1, V2 and V2.1; only the preprocessing changes.

**Selection rule:** lowest mean CV MAE; if V2.1 is within 1 EUR of the best version, prefer V2.1.

In [ ]:
prep_results, prep_city = {}, {}
for name, (prep, features) in VERSIONS.items():
    prep_results[name], prep_city[name] = evaluate(make_full_pipeline(prep, LinearRegression()), features)

fold_mae = pd.DataFrame({name: s["MAE"] for name, s in prep_results.items()})
fold_mae["V2 - V1"] = fold_mae["V2"] - fold_mae["V1"]
fold_mae["V2.1 - V2"] = fold_mae["V2.1"] - fold_mae["V2"]
fold_mae.rename(index=lambda i: f"fold {i + 1}")

In [ ]:
prep_summary = summarize(prep_results)
prep_summary

In [ ]:
pd.DataFrame(prep_city)

### Part 1 results
- **V2 vs V1:** V2 lowers MAE by about 5 EUR (≈3.4%) and is better in all 5 folds (paired differences above), mainly in Paris, London and Rome; Madrid and Barcelona get slightly worse.
- **V2.1 vs V2:** practically identical (difference ≈0.01 EUR; fold differences change sign), so the bathroom features add no measurable gain for a linear model.
- **RMSE and R²** are dominated by a few extreme prices and are not used for selection.

**Selection:** V2.1 is within the 1 EUR tolerance of the best version, so **V2.1** is selected (next cell applies the rule).

In [ ]:
best_version = prep_summary["MAE"].idxmin()
within_tolerance = prep_summary.loc["V2.1", "MAE"] - prep_summary.loc[best_version, "MAE"] < MAE_TOLERANCE
SELECTED_VERSION = "V2.1" if within_tolerance else best_version

# >>> PREPROCESSING PIPELINE INTEGRATES HERE <<<
selected_preprocessor, selected_features = VERSIONS[SELECTED_VERSION]
print(f"Lowest CV MAE: {best_version} | Selected: {SELECTED_VERSION}")

---
# Part 2 — Model Comparison
Same selected preprocessing, rows, folds, target and metrics for every model. Each model keeps its own training loss.

In [ ]:
MODELS = {
    "Dummy (median)": DummyRegressor(strategy="median"),
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
}

In [ ]:
model_results, model_city = {}, {}
for name, model in MODELS.items():
    model_results[name], model_city[name] = evaluate(make_full_pipeline(selected_preprocessor, model), selected_features)

pd.DataFrame({name: s["MAE"] for name, s in model_results.items()}).rename(index=lambda i: f"fold {i + 1}")

In [ ]:
model_summary = summarize(model_results)
model_summary["MAE vs Dummy (%)"] = (model_summary["MAE"] / model_summary.loc["Dummy (median)", "MAE"] - 1) * 100

BEST_MODEL = model_summary["MAE"].idxmin()
print(f"Strongest candidate (lowest CV MAE): {BEST_MODEL}")
model_summary.sort_values("MAE")

In [ ]:
city_table = pd.DataFrame(model_city)
city_table.loc["macro average"] = city_table.mean()
city_table

### Part 2 results
- **All trained models beat the Dummy benchmark**, so they learn real structure from the features.
- **Ridge ≈ Linear Regression:** with this much data, default regularization changes almost nothing.
- **HistGradientBoosting has the lowest CV MAE** overall, in every fold and in every city (see tables above).
- **Likely reasons (not tested separately):** trees can learn city-specific rules, use latitude/longitude as areas instead of one slope, and capture nonlinear effects such as capacity. A single linear equation cannot.
- **RMSE and R²** stay dominated by a few extreme prices.

---
# Part 3 — Conclusions

| Role | Model | Why |
|---|---|---|
| Benchmark | Dummy (median) | No-model reference |
| Baseline | Linear Regression + V2.1 | Simple, interpretable, clearly better than the benchmark |
| Strongest candidate | HistGradientBoosting + V2.1 | Lowest CV MAE overall and per city |

**Preprocessing:** V2 improved clearly on V1; V2.1 equals V2 and is kept as the shared version.

**Limitations**
- Errors are still large relative to typical prices (about 130–255 EUR).
- A few extreme prices dominate RMSE and R².
- About 56% of test listings have a host who also appears in training, so scores may be slightly optimistic.
- The candidate is not a final model: it has not been tuned or tested.

**Next steps** (training CV only): absolute-error loss, log-price target, model-specific preprocessing for boosting, light tuning. The test set is used once, for the final model.

## Prediction for a hypothetical new listing
The strongest candidate is fitted on the full training set and predicts one example listing. No test data is used.

In [ ]:
final_pipeline = make_full_pipeline(selected_preprocessor, MODELS[BEST_MODEL])
final_pipeline.fit(train_df[selected_features], train_df[TARGET])

new_listing = pd.DataFrame([{
    "city": "paris", "neighbourhood_cleansed": "Hôtel-de-Ville", "room_type": "Entire home/apt",
    "property_type": "Entire rental unit", "accommodates": 4, "bedrooms": 2, "beds": 2,
    "bathrooms": 1, "bathrooms_text": "1 bath", "latitude": 48.8566, "longitude": 2.3522,
    "minimum_nights": 2, "review_scores_rating": 4.8, "number_of_reviews": 25,
    "host_is_superhost": "f", "calculated_host_listings_count": 1, "hosts_time_as_host_years": 3,
}])

predicted = np.clip(final_pipeline.predict(new_listing[selected_features]), 0, None)[0]
print(f"{BEST_MODEL} + {SELECTED_VERSION}: predicted nightly price = {predicted:,.0f} EUR")

## Instructor checklist

| Question | Answer |
|---|---|
| Reusable working baseline? | Yes. Shared split + `preprocessing.py` + one pipeline and evaluation function; any model or preprocessing version plugs in. |
| At least one working model? | Yes. Four models run end to end; the trained ones beat the Dummy benchmark. |
| Reproducible? | Yes. Saved split IDs, fixed folds and seeds (`random_state=42`). |
| Fair evaluation? | Yes. Same rows, folds, target, features per part, preprocessing inside the pipeline, same clipping policy; test set unused. |
| Same metric? | Yes. MAE in EUR (primary), RMSE and R², overall and per city. |
| Do we understand the results? | Yes. Boosting captures city-specific and nonlinear patterns; extreme prices explain unstable RMSE/R²; host overlap may make scores slightly optimistic. |